In [ ]:
import os
import sys
sys.path.append('../src')

import pandas as pd

from preprocessing import *
from models import *

In [ ]:
project_path = os.path.dirname(os.getcwd())
dfs = get_dfs(project_path)
print(dfs['baseline_parameters'].columns)
nan_counts = dfs['baseline_parameters'].isna().sum()
nan_counts

In [ ]:
# HLA information
for col in ['MMA_broad', 'MMB_broad', 'MMDR_broad', 'MM_broad']:
    print(f"{col} - Value Counts:\n{dfs['baseline_parameters'][col].value_counts()}\nNaN Count: {dfs['baseline_parameters'][col].isna().sum()}\n")


In [ ]:
filtered_df = dfs['biopsy'].dropna(subset=['Banff 17 categorie'])

# Count occurrences per patient
banff_counts_per_patient = filtered_df.groupby('PatientID')['Banff 17 categorie'].value_counts().unstack(fill_value=0)
print(banff_counts_per_patient)
dfs['biopsy']['Banff 17 categorie'].value_counts()
print(f"unique patients {filtered_df['PatientID'].nunique()}")

In [ ]:
print(dfs['baseline_parameters']['PatientID'].nunique())
print(dfs['baseline_parameters']['TransplantationID'].nunique())

In [ ]:
print(dfs['donorparameters'].columns)
nan_counts = dfs['donorparameters'].isna().sum()
nan_counts

In [ ]:
# processing static features of patients

print(f"transplantations in baseline: {dfs['baseline_parameters']['PatientID'].count()}, unique patients: {dfs['baseline_parameters']['PatientID'].nunique()}\n")

print("remove patients where patientID, spenderID or transplantationID is NaN")
static_df = dfs['baseline_parameters'][['PatientID', 'TransplantationID', 'SpenderID', 'Geburtsdatum', 'Todesdatum', 'Datum', 'Geschlecht', 'Grunderkrankung', 'Datum_erste_Dialyse', 'Dialyse_Anzahl', 'Alter', 'Blutgruppe', 'Koerpergroesse', 'Date of graft loss', 'Loss cause', 'cold ischemia time']].rename(
    columns={
        'PatientID': 'patient_id',
        'SpenderID': 'donor_id',
        'TransplantationID': 'transplant_id',
        'Geburtsdatum': 'birth_date',
        'Todesdatum': 'death_date',
        'Datum': 'transplant_date',
        'Geschlecht': 'gender',
        'Grunderkrankung': 'underlying_disease',
        'Datum_erste_Dialyse': 'first_dialysis_date',
        'Dialyse_Anzahl': 'number_dialyses',
        'Alter': 'age',
        'Blutgruppe': 'blood_group',
        'Koerpergroesse': 'height',
        'Date of graft loss': 'loss_date',
        'Loss cause': 'loss_cause',
        'cold ischemia time': 'cold_ischemia_time',
    }
)
static_df = static_df.dropna(subset=['patient_id', 'donor_id', 'transplant_id'])

# remove duplicate patient ids
static_df = static_df.drop_duplicates(subset='patient_id', keep='first')

print(f"transplantations in baseline: {static_df['patient_id'].count()}, unique patients: {static_df['patient_id'].nunique()}")
print(static_df.columns)

In [ ]:
print("match transplantation patients to their donors")
static_df = static_df.merge(
    dfs['donorparameters'][['PatientID', 'TransplantationID', 'SpenderID', 'gender_donor', 'age_donor', 'donor_bloodgroup',
                            'type_of_donation', 'donor_COD', 'donor_weight', 'donor_height', 'SOURCE']],
    how='left',  # Left join to keep all records in static_df
    left_on=['patient_id', 'transplant_id', 'donor_id'],
    right_on=['PatientID', 'TransplantationID', 'SpenderID']
)

static_df = static_df.drop(columns=['PatientID', 'TransplantationID', 'SpenderID'])

print(f"transplantations in baseline: {static_df['patient_id'].count()}, unique patients: {static_df['patient_id'].nunique()}")
print(static_df.columns)
print(static_df.isna().sum())

In [ ]:
static_df['transplant_date'] = pd.to_datetime(static_df['transplant_date'], dayfirst=True, errors='coerce')
static_df['loss_date'] = pd.to_datetime(static_df['loss_date'], dayfirst=True, errors='coerce')
static_df['death_date'] = pd.to_datetime(static_df['death_date'], dayfirst=True, errors='coerce')

# if loss date is given calculate the relative days from transplantation date
static_df['loss_rel_days'] = (static_df['loss_date'] - static_df['transplant_date']).dt.days
static_df['death_rel_days'] = (static_df['death_date'] - static_df['transplant_date']).dt.days

In [ ]:
# selecting relevant features
static_df = static_df[
    [
        'patient_id',
        'transplant_id',
        'transplant_date',
        'gender',
        'age',
        'death_rel_days',
        'underlying_disease',
        'number_dialyses',
        'blood_group',
        'height',
        'loss_rel_days',
        'loss_cause',
        'cold_ischemia_time',
        'gender_donor',
        'age_donor',
        'donor_bloodgroup',
        'type_of_donation',
        'donor_height',
        'donor_weight'
    ]
]

In [ ]:
# creating static df using the function
project_path = os.path.dirname(os.getcwd())
dfs = get_dfs(project_path)
stat = create_static_df(dfs)
stat